In [14]:
import numpy as np
import pandas as pd
import re
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [15]:
df = pd.read_csv("sqliv2.csv",encoding="utf-16")  # غير الاسم لو مختلف

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'sqliv2.csv'

In [ ]:
def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s\(\)=\'\"\-\_\*\+\,\.]', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

# ناخد بس الهجمات
texts = df[df['Label'] == 1]['Sentence'].apply(clean_text)

texts = texts.dropna()
texts = texts[texts.str.len() > 10]

texts.head()

In [ ]:
texts = ["<start> " + t + " <end>" for t in texts]

In [ ]:
seq_length = 40

X = []
y = []

for i in range(len(text) - seq_length):
    seq = text[i:i+seq_length]
    label = text[i+seq_length]
    
    X.append([char_to_idx[c] for c in seq])
    y.append(char_to_idx[label])

X = np.array(X)
y = np.array(y)

# reshape + normalize
X = X.reshape((X.shape[0], X.shape[1], 1))
X = X / float(vocab_size)

print(X.shape, y.shape)

In [ ]:
model = Sequential()

model.add(LSTM(256, return_sequences=True, input_shape=(seq_length, 1)))
model.add(Dropout(0.2))

model.add(LSTM(256, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(512, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(64))  # آخر واحدة بس تكون False
model.add(Dropout(0.2))

model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

model.summary()

In [ ]:
model.fit(X, y, epochs=30, batch_size=16)

In [ ]:
def sample_top_k(preds, k=5):
    preds = np.asarray(preds).astype('float64')
    
    top_k_indices = preds.argsort()[-k:]
    top_k_probs = preds[top_k_indices]
    top_k_probs = top_k_probs / np.sum(top_k_probs)
    
    return np.random.choice(top_k_indices, p=top_k_probs)

In [ ]:
def generate_text(model, length=120):
    start_index = random.randint(0, len(text) - seq_length - 1)
    pattern = text[start_index:start_index + seq_length]
    
    generated = pattern
    
    for _ in range(length):
        x = np.array([char_to_idx[c] for c in pattern])
        x = x.reshape((1, seq_length, 1))
        x = x / float(vocab_size)
        
        prediction = model.predict(x, verbose=0)[0]
        
        index = sample_top_k(prediction, k=5)
        result = idx_to_char[index]
        
        generated += result
        pattern = pattern[1:] + result

        # منع التكرار الغبي
        if len(set(generated[-10:])) == 1:
            break
    
    return generated

In [ ]:
for _ in range(5):
    print("----")
    print(generate_text(model))

In [ ]:
for _ in range(5):
    print("----")
    print(generate_text(model, 120))

In [ ]:
def clean_generated(text):
    allowed = "abcdefghijklmnopqrstuvwxyz0123456789 ()=,'-*+._"
    return ''.join([c for c in text if c in allowed])

for _ in range(5):
    print("----")
    print(clean_generated(generate_text(model)))